In [ ]:
!pip install eyepop==3.12.0
!pip install google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.6/89.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 76.8 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 49.0.0
    Uninstalling cryptography-49.0.0:
      Successfully uninstalled cryptography-49.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.3.0 requires cryptography<50,>=49.0.0, but you have cryptography 46.0.7 which is incompatible.


In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

Enter your Account UUID: a5184defa8e847248f589d35080efbfa
Enter your API KEY: ··········


In [ ]:
GEMINI_API_KEY=getpass.getpass('Enter yofur API KEY: ')

Enter yofur API KEY: ··········


In [ ]:
NAMESPACE_PREFIX="datasciencealliance-org" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import InferenceComponent, Pop
import json

retail_shelf_prompt = """
  Determine the primary content of the image and assign exactly one label: ['Empty', 'Out_of_Stock', 'OK'].

  Choose 'Empty' only if the image's main focus is a retail shelf slot, shelf bay, or assigned product area with zero visible units of the target product remaining. The target shelf space must be completely bare. If there is even one visible unit of the target product remaining in the main target shelf area, do not choose 'Empty'.

  Choose 'Out_of_Stock' if the image's main focus is a retail shelf slot, shelf bay, or assigned product area where at least one visible unit of the target product remains, but the shelf space is clearly low inventory. This includes cases where only 1 to 5 units remain, products are sparse, pushed to one side, widely separated, isolated in a mostly empty shelf bay, or surrounded by large empty gaps where more of the same product should be. If the target area is mostly empty but still has at least one product unit remaining, choose 'Out_of_Stock' instead of 'Empty'.

  Choose 'OK' only if the image's main focus is a retail shelf slot, shelf bay, or assigned product area that is sufficiently stocked with the target product. Products should be visible across most of the target shelf space, with no large obvious gaps and no immediate restocking needed.

  Ignore neighboring stocked shelves, background products, price tags, shelf dividers, shadows, reflections, advertisements, and signs unless they are part of the main target product area. Do not rely on written words like “empty” or “out of stock”; classify based on the visible shelf condition.

  Return only the single best-fitting label.
"""

ability_prototypes = [
  VlmAbilityCreate(
  name=f"{NAMESPACE_PREFIX}.image-classify.retail-shelf-stock",
  description="Classify a retail shelf image as empty, out of stock / low inventory, or OK",
  worker_release="qwen3-instruct",
  text_prompt=retail_shelf_prompt,
  transform_into=TransformInto(),
  config=InferRuntimeConfig(
  max_new_tokens=10,
  image_size=640
  ),
  is_public=False
  )
]


### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

created ability 06a397e8b674720f8000fa1e8b20c4ad with alias entries [AbilityAliasEntry(alias='datasciencealliance-org.image-classify.retail-shelf-stock', tag='1.0.2'), AbilityAliasEntry(alias='datasciencealliance-org.image-classify.retail-shelf-stock', tag='latest')]


### Evalulate on a Single Image

In [ ]:
from pathlib import Path
import json
from eyepop import EyePopSdk
from eyepop.worker.worker_types import InferenceComponent, Pop

pop = Pop(components=[
    InferenceComponent(
        ability=f"{NAMESPACE_PREFIX}.image-classify.retail-shelf-stock:latest"
    )
])

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
    endpoint.set_pop(pop)

    sample_img_path = Path("/content/sample_img.jpg")
    job = endpoint.upload(sample_img_path)

    while result := job.predict():
        print(json.dumps(result, indent=2))

print("Done")

{
  "compute_units": 0.1,
  "seconds": 0,
  "source_height": 1024,
  "source_id": "2f52cf1e-6e68-11f1-b452-7a8ddcd9c1e7",
  "source_width": 1024,
  "system_timestamp": 1782152920374648000,
  "texts": [
    {
      "id": 1,
      "text": "Out_of_Stock"
    }
  ],
  "timestamp": 0
}
Done
